In [1]:
from database import init_db, get_connection

In [4]:

init_db()

In [11]:
def criar_cliente(nome: str, senha: str, saldo_inicial: float = 0.0) -> int:
    with get_connection() as conn:
        cursor = conn.cursor()
        
        # Inserir o cliente
        cursor.execute(
            "INSERT INTO clientes (nome, senha, saldo) VALUES (?, ?, ?)",
            (nome, senha, saldo_inicial)
        )
        id_cliente = cursor.lastrowid
        
        # Criar a carteira vinculada ao cliente
        cursor.execute(
            "INSERT INTO carteiras (id_cliente) VALUES (?)",
            (id_cliente,)
        )
        
        conn.commit()  # Garante a persistência das alterações
        return id_cliente

def buscar_cliente_por_id(id_cliente: int):
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(
            "SELECT nome, saldo FROM clientes WHERE id_cliente = ?",
            (id_cliente,)
        )
        cliente = cursor.fetchone()
        
        if cliente:
            print(f"Nome: {cliente['nome']} | Saldo: R${cliente['saldo']:.2f}")
            return dict(cliente)
        return None

def adicionar_saldo(id_cliente: int, valor: float):
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute(
            "UPDATE clientes SET saldo = saldo + ? WHERE id_cliente = ?",
            (valor, id_cliente)
        )
        conn.commit()


def adicionar_ativos(id_carteira: int, ticker: str, qtd: int):
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor = conn.execute(
            """INSERT INTO ativos_carteira (id_carteira, ticker, quantidade) VALUES (?, ?, ?) 
            ON CONFLICT(id_carteira, ticker) DO UPDATE SET quantidade = quantidade + excluded.quantidade""",
            (id_carteira, ticker.strip().upper(), qtd)
        )
        conn.commit()
        return "deu bom?"

def buscar_ativos_carteira_por_id(id_carteira: int):
    with get_connection() as conn:
            cursor = conn.cursor()
            cursor.execute(
                "SELECT ticker, quantidade FROM ativos_carteira WHERE id_carteira = ?",
                (id_carteira, )
            )
            ativos = cursor.fetchall()
            
            if ativos:
                return [dict(ativo) for ativo in ativos]
            return []

    

In [5]:
# Exemplo de uso:
novo_id = criar_cliente("EdsonEdu", "2031", 1000.0)
print(f"Cliente criado com ID: {novo_id}")

Cliente criado com ID: 1


In [6]:
buscar_cliente_por_id(1)

Nome: EdsonEdu | Saldo: R$1000.00


{'nome': 'EdsonEdu', 'saldo': 1000.0}

In [14]:
adicionar_ativos(1, "abba", 12)

'deu bom?'

In [19]:
a = buscar_ativos_carteira_por_id(1)
for ativo in a:
    print(f"ticker: {ativo['ticker']}, quantidade: {ativo["quantidade"]}")

ticker: ABBA, quantidade: 12
ticker: FDS, quantidade: 234
